# CS 4412 – Data Mining
## Milestone 3 / 4: MLB Player Aging Analysis (Final)
Tate York

This notebook is the complete, end-to-end analysis pipeline for the CS 4412 project.
It builds a player-season dataset from the Lahman Baseball Database (1980-2025),
applies K-Means clustering to identify offensive profiles, examines aging curves by
profile, and uses a decision tree as an interpretability tool. The Milestone 4 pass
adds consistent figure styling, a seed-stability test for the clustering, and an
explicit validity, limitations, and ethics section.

Sections:

1. Setup and styling
2. Loading the Lahman tables
3. Dataset construction and feature engineering
4. Exploratory data analysis
5. Clustering: cluster selection, model fit, profile interpretation
6. Aging curves by offensive profile
7. Decision tree interpretability
8. Seed-stability test
9. Validity, limitations, and ethical considerations
10. Summary

## 1. Setup and Styling

A consistent matplotlib style is applied to every figure for a portfolio-quality
presentation. A fixed color palette is also reserved for the four named offensive
profiles so the same color represents the same profile across plots.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Consistent figure styling for the whole notebook
sns.set_theme(context="notebook", style="whitegrid", palette="deep")
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "legend.fontsize": 10,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
})

# Reserved palette for the named offensive profiles
PROFILE_ORDER = ["power", "balanced", "contact", "low-production"]
PROFILE_COLORS = {
    "power":          "#d62728",   # red
    "balanced":       "#1f77b4",   # blue
    "contact":        "#2ca02c",   # green
    "low-production": "#7f7f7f",   # grey
}

print("Python:", sys.executable.split('/')[-1])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 2. Loading the Lahman Tables

The Lahman Baseball Database supplies historical season-level batting statistics
and player demographic information. Two tables are used:

- `Batting.csv` — one row per player-season-team
- `People.csv` — player demographics including birth year

The raw files are not committed to the repository; instructions for downloading
them are in `README.md`.

In [ ]:
lahman_dir = Path("../data/raw/lahman")

batting = pd.read_csv(lahman_dir / "Batting.csv")
people  = pd.read_csv(lahman_dir / "People.csv")

print(f"Batting: {batting.shape[0]:,} rows x {batting.shape[1]} cols")
print(f"People:  {people.shape[0]:,} rows x {people.shape[1]} cols")
batting.head()

## 3. Dataset Construction and Feature Engineering

A clean player-season dataset is built by:

1. Filtering the Batting table to the modern era (`yearID` between 1980 and 2025).
2. Aggregating multi-team seasons into a single row per `(playerID, yearID)`.
3. Joining birth year from People to compute `Age = yearID - birthYear`.
4. Computing rate statistics: `BA`, `OBP`, `SLG`, `OPS`.
5. Computing a season-and-league-adjusted `OPS_plus_lg` = `100 * OPS / LgAvgOPS`.
6. Filtering to player-seasons with at least 200 plate appearances to reduce
   noise from very small samples.

In [ ]:
# Filter to the modern era
bat = batting[batting["yearID"].between(1980, 2025)].copy()

# Ensure required columns exist (older Lahman dumps occasionally omit HBP / SF)
needed = ["playerID","yearID","lgID","G","AB","H","2B","3B","HR","BB","SO","HBP","SF"]
for c in needed:
    if c not in bat.columns:
        bat[c] = 0
bat = bat[needed].copy()

# Combine multi-team seasons into one row per player-season
sum_cols = ["G","AB","H","2B","3B","HR","BB","SO","HBP","SF"]
ps = bat.groupby(["playerID","yearID","lgID"], as_index=False)[sum_cols].sum()

# Merge birth year to compute player age
ps = ps.merge(people[["playerID","birthYear"]], on="playerID", how="left")
ps["Age"] = ps["yearID"] - ps["birthYear"]

# Plate appearances and rate statistics
ps["PA"]  = ps["AB"] + ps["BB"] + ps["HBP"] + ps["SF"]
ps["BA"]  = ps["H"] / ps["AB"]
ps["OBP"] = (ps["H"] + ps["BB"] + ps["HBP"]) / (ps["AB"] + ps["BB"] + ps["HBP"] + ps["SF"])
ps["SLG"] = (ps["H"] + ps["2B"] + 2*ps["3B"] + 3*ps["HR"]) / ps["AB"]
ps["OPS"] = ps["OBP"] + ps["SLG"]

# Drop missing values and apply sample-size filter
ps = ps.dropna(subset=["Age","OPS","OBP","SLG","BA","PA"])
ps = ps[(ps["Age"] > 0) & (ps["Age"] < 60)]
ps = ps[ps["PA"] >= 200]

ps.rename(columns={"yearID":"Season","lgID":"Lg"}, inplace=True)
print(f"Player-seasons after PA >= 200 filter: {len(ps):,}")
ps.head()

In [ ]:
# Season-and-league average OPS for league adjustment
lg_ops = ps.groupby(["Season","Lg"], as_index=False)["OPS"].mean()
lg_ops = lg_ops.rename(columns={"OPS":"LgAvgOPS"})
ps = ps.merge(lg_ops, on=["Season","Lg"], how="left")
ps["OPS_plus_lg"] = 100 * (ps["OPS"] / ps["LgAvgOPS"])
ps[["playerID","Season","Age","PA","OPS","LgAvgOPS","OPS_plus_lg"]].head()

## 4. Dataset Summary

After preprocessing, each row represents a single player-season.

In [ ]:
df = ps.copy()

# Persist the cleaned dataset for downstream reuse
out_path = Path("../data/mlb_player_seasons_1980_2025_lahman_batting.csv")
df.to_csv(out_path, index=False)

print(f"Dataset:        {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory:         {df.memory_usage(deep=True).sum() / 2**20:.2f} MB")
print(f"Unique players: {df['playerID'].nunique():,}")
print(f"Season range:   {int(df['Season'].min())}–{int(df['Season'].max())}")
df.sample(8, random_state=42)

## 5. Exploratory Data Analysis

Three lightweight EDA views motivate the rest of the analysis: how player ages
are distributed, how mean offensive output evolves with age, and how offensive
metrics correlate with each other.

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["Age"], bins=25, color="#4C78A8", edgecolor="white")
ax.set_title("Distribution of Player Ages (1980–2025)")
ax.set_xlabel("Age")
ax.set_ylabel("Number of Player-Seasons")
plt.show()

In [ ]:
age_perf = df.groupby("Age")["OPS"].mean()

fig, ax = plt.subplots()
ax.plot(age_perf.index, age_perf.values, marker="o", color="#4C78A8")
ax.set_title("Average OPS by Age (Population)")
ax.set_xlabel("Age")
ax.set_ylabel("Mean OPS")
plt.show()

In [ ]:
players_by_age = df.groupby("Age")["playerID"].nunique()

fig, ax = plt.subplots()
ax.plot(players_by_age.index, players_by_age.values, marker="o", color="#54A24B")
ax.set_title("Number of Active Players by Age")
ax.set_xlabel("Age")
ax.set_ylabel("Unique Players")
plt.show()

In [ ]:
features = ["PA","HR","BB","SO","BA","OBP","SLG","OPS","OPS_plus_lg"]
corr = df[features].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation Between Offensive Metrics")
plt.show()

## 6. Clustering: Offensive Profiles

K-Means clustering is applied to standardized offensive features in order to
discover natural groupings of player-seasons by hitting style and productivity.

Features used for clustering:
`BA`, `OBP`, `SLG`, `HR`, `SO`, and `OPS_plus_lg`.

These metrics capture both rate-style production (BA, OBP, SLG, OPS+) and
volume / approach indicators (HR, SO), giving the algorithm enough signal to
separate power, contact, balanced, and low-production profiles.

### 6.1 Choosing the Number of Clusters

Milestone 2 used `k = 4` without formal justification. To justify the choice,
the elbow method (inertia) and silhouette score are evaluated for `k` in 2–8
on the standardized feature matrix.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_features = ["BA", "OBP", "SLG", "HR", "SO", "OPS_plus_lg"]
X = df[cluster_features].dropna().copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias, sil_scores = [], []
k_values = list(range(2, 9))

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_values, inertias, marker="o", color="#4C78A8")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")

axes[1].plot(k_values, sil_scores, marker="o", color="#F58518")
axes[1].set_title("Silhouette Scores")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette")

plt.tight_layout()
plt.show()

for k, s in zip(k_values, sil_scores):
    print(f"k={k}, silhouette={s:.4f}")

Although the silhouette score is highest at `k = 2`, that solution merges
distinct hitting styles into two very broad groups. The elbow plot bends
between `k = 3` and `k = 4`, and `k = 4` produces clusters that map cleanly
onto recognizable offensive profiles (**power**, **balanced**, **contact**,
**low-production**). `k = 4` is therefore selected as the final number of
clusters — a balance of statistical support and baseball-specific
interpretability.

In [ ]:
def assign_profile_labels(kmeans_model, scaler, cluster_features):
    """
    Map KMeans cluster IDs to interpretable profile names by inspecting the
    cluster centroids in the original (unstandardized) feature space.

    Returns a dict {cluster_id: profile_name}.
    """
    centroids_std = kmeans_model.cluster_centers_
    centroids_orig = scaler.inverse_transform(centroids_std)
    centroid_df = pd.DataFrame(centroids_orig, columns=cluster_features)
    centroid_df["cluster"] = range(len(centroid_df))

    # Rank by overall offensive production
    ranked = centroid_df.sort_values("OPS_plus_lg", ascending=False).reset_index(drop=True)
    high_prod_ids = ranked.iloc[:2]["cluster"].tolist()
    low_prod_ids  = ranked.iloc[2:]["cluster"].tolist()

    # Within high-production: power has the most HR, balanced is the other
    power_id    = centroid_df.loc[centroid_df["cluster"].isin(high_prod_ids), "HR"].idxmax()
    balanced_id = [c for c in high_prod_ids if c != power_id][0]
    # Within low-production: contact has the higher BA, low-production is the other
    contact_id  = centroid_df.loc[centroid_df["cluster"].isin(low_prod_ids), "BA"].idxmax()
    low_id      = [c for c in low_prod_ids if c != contact_id][0]

    return {power_id: "power", balanced_id: "balanced",
            contact_id: "contact", low_id: "low-production"}


# Fit the final K-Means model
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df_cluster = df.loc[X.index].copy()
df_cluster["cluster"] = clusters

label_map = assign_profile_labels(kmeans, scaler, cluster_features)
df_cluster["profile"] = df_cluster["cluster"].map(label_map)

print("Cluster -> profile mapping:", label_map)
df_cluster[["playerID","Season","Age","OPS","OPS_plus_lg","cluster","profile"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for prof in PROFILE_ORDER:
    sub = df_cluster[df_cluster["profile"] == prof]
    ax.scatter(sub["OBP"], sub["SLG"], s=14, alpha=0.55,
               color=PROFILE_COLORS[prof], label=prof)

ax.set_title("Player-Season Clusters by Offensive Profile (k = 4)")
ax.set_xlabel("On-Base Percentage")
ax.set_ylabel("Slugging Percentage")
ax.legend(title="Profile")
plt.show()

### 6.2 Cluster Profile Interpretation

The mean of each clustering feature plus a few auxiliary stats is computed per
named profile. This is the core interpretability table — it shows, in original
units, *what each profile looks like* on the field.

In [ ]:
profile_summary = (
    df_cluster.groupby("profile")[cluster_features + ["PA", "OPS"]]
              .mean()
              .round(3)
              .reindex(PROFILE_ORDER)
)
profile_summary["count"] = df_cluster.groupby("profile").size().reindex(PROFILE_ORDER)
profile_summary

## 7. Aging Curves by Offensive Profile

Player-seasons are grouped by `(profile, Age)` and the mean OPS is computed for
each cell. Cells with fewer than 20 player-seasons are dropped to avoid
small-sample noise. The plot shows whether different offensive profiles age
along different trajectories.

In [ ]:
aging = (
    df_cluster.groupby(["profile", "Age"])
              .agg(mean_OPS=("OPS", "mean"), n=("OPS", "size"))
              .reset_index()
)
aging = aging[aging["n"] >= 20]

fig, ax = plt.subplots(figsize=(9, 6))
for prof in PROFILE_ORDER:
    sub = aging[aging["profile"] == prof].sort_values("Age")
    if sub.empty:
        continue
    ax.plot(sub["Age"], sub["mean_OPS"], marker="o",
            color=PROFILE_COLORS[prof], label=prof)

ax.set_title("Aging Curves by Offensive Profile")
ax.set_xlabel("Age")
ax.set_ylabel("Mean OPS")
ax.legend(title="Profile")
plt.show()

The four profiles remain clearly separated across age. The **power** group
maintains the highest production at every age, while the **low-production**
group sits consistently below the others. The general shape — a rise during
the early-to-mid 20s, a plateau, then a decline in the mid-30s — is shared
across profiles, but the *level* of production differs substantially.
Offensive profile therefore matters when interpreting aging.

Note that survivorship bias likely flattens late-career decline: older players
in the dataset are disproportionately the high-performers who survived long
enough to keep playing. This is discussed in the validity section.

## 8. Decision Tree Interpretability

A decision tree classifier is fit using the cluster assignments as class labels.
Predictive accuracy is *not* the goal — the tree is used as an interpretability
tool. The split features and feature-importance scores reveal which offensive
variables most cleanly separate the four profiles.

A 75/25 stratified train/test split is used so that the train and test
accuracies can be compared as a basic sanity check that the structure the tree
is learning generalizes.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_tree = df_cluster[cluster_features]
y_tree = df_cluster["cluster"]

X_train, X_test, y_train, y_test = train_test_split(
    X_tree, y_tree, test_size=0.25, random_state=42, stratify=y_tree
)

# Shallow tree for interpretability rather than predictive depth
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)

train_acc = accuracy_score(y_train, tree.predict(X_train))
test_acc  = accuracy_score(y_test,  tree.predict(X_test))
print(f"Decision tree (max_depth=4) train accuracy: {train_acc:.3f}")
print(f"Decision tree (max_depth=4) test  accuracy: {test_acc:.3f}")

In [ ]:
importances = (
    pd.Series(tree.feature_importances_, index=cluster_features)
      .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importances.index, importances.values, color="#4C78A8")
ax.set_title("Decision Tree Feature Importance")
ax.set_xlabel("Importance")
plt.show()

importances.sort_values(ascending=False).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    tree,
    feature_names=cluster_features,
    class_names=[f"cluster {c}" for c in sorted(df_cluster['cluster'].unique())],
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3,
    ax=ax,
)
ax.set_title("Decision Tree (top three levels)")
plt.show()

The feature importances confirm the structure of the clusters:
`OPS_plus_lg` is the dominant variable separating profiles, with `HR` and
`SO` contributing meaningfully as secondary splits. Rate statistics
(`BA`, `OBP`, `SLG`) carry less standalone weight in the tree because their
information is largely absorbed by `OPS_plus_lg`.

Train and test accuracy are very close, which is expected: the tree is
predicting the cluster labels that K-Means produced from the same features,
so the supervised problem is well-defined and easy. The point of this step
is the *interpretation* — splits and importances — not the accuracy number.

## 9. Seed-Stability of the Clustering

K-Means is sensitive to initialization. To check that the four-cluster
structure is stable across random seeds, the model is refit with several
seeds and each new label vector is compared against the original (`seed=42`)
solution using the **Adjusted Rand Index** (ARI).

ARI is permutation-invariant — it does not care which integer label K-Means
happens to assign to "power" — so values close to 1.0 indicate that the same
*partition* of player-seasons is being recovered each time.

In [ ]:
from sklearn.metrics import adjusted_rand_score

reference_labels = df_cluster["cluster"].values
seeds = [0, 7, 13, 21, 99]

stability = []
for seed in seeds:
    km_alt = KMeans(n_clusters=4, random_state=seed, n_init=10).fit(X_scaled)
    ari = adjusted_rand_score(reference_labels, km_alt.labels_)
    stability.append({"seed": seed, "ARI_vs_seed42": round(ari, 4)})

stability_df = pd.DataFrame(stability)
print("Stability of the 4-cluster solution across random seeds:")
print(stability_df.to_string(index=False))

print(f"\nMean ARI: {stability_df['ARI_vs_seed42'].mean():.3f}")
print(f"Min  ARI: {stability_df['ARI_vs_seed42'].min():.3f}")

High ARI values (close to 1.0) indicate that the four-cluster partition is
recovered consistently across seeds, which supports treating the
`power / balanced / contact / low-production` structure as a stable property
of the data rather than an artifact of one lucky initialization.

## 10. Validity, Limitations, and Ethical Considerations

### Validity

- **Cluster stability.** The seed-stability test (Section 9) shows the
  four-cluster partition is recovered consistently across initializations,
  so the structure is not an artifact of one random seed.
- **Decision tree generalization.** Train and test accuracy on the cluster
  labels are very close (Section 8), indicating the tree is learning genuine
  structural separations rather than memorizing the training set.
- **Cluster selection.** `k = 4` was chosen with explicit elbow-and-silhouette
  evaluation and a stated interpretability tradeoff against the silhouette-
  optimal `k = 2`. Reported with that caveat.
- **Alternative explanations.** The cluster split is dominated by overall
  production (`OPS_plus_lg`) rather than playing style. A skeptical reading
  is that the model is partly recovering "good hitter / bad hitter" tiers
  rather than four genuinely distinct hitting styles.

### Limitations

- **Survivorship bias.** Older player-seasons are disproportionately drawn
  from above-average hitters who survived long enough to keep playing.
  This likely *flattens* late-career decline in the aging curves and means
  the curves should be read as conditional on continued MLB tenure, not as
  an unconditional aging effect.
- **Sample-size filter.** Player-seasons with `PA < 200` are excluded.
  This stabilizes rate statistics but also removes part-time players, late
  call-ups, and end-of-career role players whose aging trajectories may
  differ from the included population.
- **Approximated league adjustment.** `OPS_plus_lg = 100 * OPS / LgAvgOPS`
  is a simple league-average adjustment, not the full park-and-league-
  adjusted Baseball Reference OPS+. Park effects are not removed.
- **Feature scope.** Defense, baserunning, position, handedness, and
  injury are not in the feature set. The "profiles" identified are
  *offensive* profiles only.
- **Era effects.** The 1980-2025 window spans clear scoring-environment
  shifts (steroid era, post-2010 strikeout boom). League adjustment
  partially controls for this, but era-specific structural breaks are
  not modeled directly.
- **Generalizability.** Results describe MLB player-seasons since 1980.
  They do not necessarily generalize to other leagues, eras, levels of
  play, or non-baseball aging contexts.

### Ethical Considerations

- **Privacy.** The dataset is public, derived from box-score and roster
  records, and contains no sensitive personal information.
- **Fairness and misuse.** The intended audience is academic — the project
  is exploratory pattern discovery, not player evaluation for contracts,
  arbitration, or roster decisions. Feature importance results should
  *not* be interpreted as a recommendation for who teams should sign or
  cut. Decisions about real players' careers should integrate domain
  expertise, scouting, defense, health, and context that this analysis
  does not include.
- **Bias in interpretation.** Names like *low-production* are convenient
  technical shorthands; in any portfolio or presentation context they
  should be framed as descriptions of the cluster's offensive profile in
  this dataset, not as judgments about individual players.

## 11. Summary

This notebook delivers an end-to-end data mining pipeline for MLB
offensive performance and aging:

1. **Dataset construction** — player-season records from the Lahman Batting
   and People tables (1980–2025), filtered to seasons with at least 200
   plate appearances.
2. **Feature engineering** — BA, OBP, SLG, OPS, and league-adjusted
   `OPS_plus_lg` per player-season.
3. **Exploratory analysis** — age distribution, mean OPS by age, active
   players by age, and a correlation heatmap of offensive metrics.
4. **Cluster selection** — elbow + silhouette evaluated for `k` in 2–8
   with `k = 4` chosen for interpretability.
5. **Cluster interpretation** — four named offensive profiles
   (power, balanced, contact, low-production) derived deterministically
   from cluster centroids.
6. **Aging curves by profile** — separate trajectories computed for each
   cluster, showing that profiles remain separated across age.
7. **Decision tree interpretability** — feature importance shows
   `OPS_plus_lg`, `HR`, and `SO` are most responsible for separating the
   four profiles.
8. **Seed-stability test** — Adjusted Rand Index across multiple random
   seeds confirms the four-cluster partition is stable.
9. **Validity, limitations, ethics** — explicit treatment of
   survivorship bias, sample-size filtering, league-adjustment
   approximation, and appropriate use.

All visualizations share a consistent style and color palette so the same
profile is the same color across plots.